# Неделя 1. Введение в RL: зачем это всё нужно, бандиты, MDP

Курс Reinforcement Learning. Вводная лекция.

**План на сегодня**

1. Как устроен курс
2. Зачем нужно обучение с подкреплением: примеры, которые изменили область
3. Что такое RL и чем оно отличается от «обычного» машинного обучения
4. Живое демо: агент в среде Gymnasium
5. Из чего состоит задача RL: состояние, действие, награда, политика, return
6. Главная дилемма: exploration vs exploitation (многорукие бандиты)
7. Формализация: марковский процесс принятия решений и уравнения Беллмана
8. Карта курса и литература

> Ноутбук рассчитан на живой показ: код-ячейки короткие и выполняются на CPU за секунды.
> Перед занятием стоит один раз прогнать все ячейки и открыть ссылки на видео во вкладках.

## 0. Как устроен курс

* **15 недель**, одно занятие в неделю: лекция + семинар. Каждая неделя — папка `NN-topic-name/` в репозитории с тремя ноутбуками: `lecture/`, `seminar/`, `homework/`.
* **Домашние задания** почти каждую неделю, в `.ipynb`. Часть проверок — через `assert`, так что можно проверить себя до сдачи.
* **Оценка** (черновик, обсуждаем сегодня): 60% домашние задания, 30% итоговый проект, 10% активность на семинарах.
* **Итоговый проект**: своя реализация RL-агента или мини-исследование на среде по выбору. Темы появятся к 8-й неделе.
* **Инструменты**: Python 3.10+, [Gymnasium](https://gymnasium.farama.org/) (среды), [PyTorch](https://pytorch.org/) (нейросети), Jupyter.
  Библиотеки [Stable-Baselines3](https://stable-baselines3.readthedocs.io/) и [CleanRL](https://github.com/vwxyzjn/cleanrl) читаем как справочник, но алгоритмы в домашках пишем сами.
* **Что нужно уметь на входе**: линейная алгебра, теория вероятностей, градиентный спуск, Python с numpy. Нейросети и PyTorch — желательно, но необходимый минимум разберём на мини-семинаре `../seminar/pytorch_intro.ipynb`.

Установка окружения:

```bash
python -m venv .venv && source .venv/bin/activate
pip install -r requirements.txt
```

## 1. Зачем это всё нужно

Обучение с подкреплением — это про **принятие последовательных решений**: не «что на картинке?», а «что делать сейчас, чтобы через сто шагов было хорошо?». Такие задачи повсюду: игры, роботы, управление инфраструктурой, диалоговые модели.

Ниже несколько историй, с которых обычно начинают. У каждой есть ссылка на демо или статью: посмотрите видео, это лучшая мотивация.

### Игры: от нард до StarCraft

| Год | Система | Что произошло | Посмотреть |
|---|---|---|---|
| 1992 | **TD-Gammon** (Tesauro) | Нейросеть + TD-обучение играет в нарды на уровне чемпионов мира. Первый большой успех RL. | [статья](https://dl.acm.org/doi/10.1145/203330.203343) |
| 2013–2015 | **DQN** (DeepMind) | Одна и та же сеть учится играть в 49 игр Atari, глядя только на пиксели и счёт. | [видео Breakout](https://www.youtube.com/watch?v=TmPfTpjtdgg), [Nature](https://www.nature.com/articles/nature14236) |
| 2016 | **AlphaGo** | Победа над Ли Седолем в го. За год до этого считалось, что до этого ещё десятилетие. | [фильм AlphaGo](https://www.youtube.com/watch?v=WXuK6gekU1Y), [страница проекта](https://deepmind.google/research/breakthroughs/alphago/) |
| 2017 | **AlphaZero** | Тот же алгоритм, обучаясь только игрой с самим собой, осваивает шахматы, сёги и го с нуля. | [блог DeepMind](https://deepmind.google/discover/blog/alphazero-shedding-new-light-on-chess-shogi-and-go/) |
| 2019 | **OpenAI Five**, **AlphaStar** | Победы над профессионалами в Dota 2 и StarCraft II: длинные горизонты, частичная наблюдаемость, командная игра. | [OpenAI Five](https://openai.com/index/openai-five/), [AlphaStar](https://deepmind.google/discover/blog/alphastar-mastering-the-real-time-strategy-game-starcraft-ii/) |
| 2019 | **Hide and Seek** (OpenAI) | Агенты в прятках сами изобретают использование инструментов: строят укрытия, «сёрфят» на ящиках. | [блог с гифками](https://openai.com/index/emergent-tool-use/) |

### Робототехника

* [Роботы-футболисты DeepMind](https://sites.google.com/view/op3-soccer): походка, удары, подъём после падения — всё выучено в симуляторе и перенесено на железо (sim-to-real).
* [Обучение ходьбе ANYmal](https://arxiv.org/abs/1901.08652) и [«научиться ходить за минуты»](https://arxiv.org/abs/2109.11978): RL-контроллеры для четвероногих роботов, которые устойчивее ручных.
* [Boston Dynamics: RL для Spot](https://bostondynamics.com/blog/starting-on-the-right-foot-with-reinforcement-learning/): коммерческий робот, часть контроллеров которого теперь обучается, а не программируется.
* [Кубик Рубика одной рукой](https://openai.com/index/solving-rubiks-cube/): пример, насколько трудно перенести политику из симуляции в реальный мир.

### Инфраструктура и наука

* [Охлаждение дата-центров Google](https://deepmind.google/discover/blog/deepmind-ai-reduces-google-data-centre-cooling-bill-by-40/): минус 40% энергии на охлаждение.
* [AlphaTensor](https://deepmind.google/discover/blog/discovering-novel-algorithms-with-alphatensor/) и [AlphaDev](https://deepmind.google/discover/blog/alphadev-discovers-faster-sorting-algorithms/): RL находит новые алгоритмы умножения матриц и сортировки.
* Управление плазмой в токамаке, планирование чипов, рекомендательные системы, где важно долгосрочное вовлечение, а не один клик.

### Языковые модели

* **RLHF** ([InstructGPT, 2022](https://arxiv.org/abs/2203.02155)): ChatGPT стал «полезным собеседником» именно благодаря RL на человеческих предпочтениях. Разберём на неделе 14.
* **Рассуждающие модели** ([DeepSeek-R1, 2025](https://arxiv.org/abs/2501.12948)): RL на проверяемых наградах (правильно ли решена задача) учит модель длинным цепочкам рассуждений.

Общий сюжет: как только у задачи есть **среда** и **числовая цель**, RL позволяет не программировать поведение вручную, а выучить его.

![timeline](../../assets/rl_timeline.png)

### Интерактивные демо для домашнего «потыкать»

* [ReinforceJS](https://cs.stanford.edu/people/karpathy/reinforcejs/) (A. Karpathy): GridWorld с динамическим программированием и TD-обучением прямо в браузере, PuckWorld, WaterWorld с DQN. Отличная иллюстрация к неделям 2–5.
* [Каталог сред Gymnasium](https://gymnasium.farama.org/environments/classic_control/): гифки всех стандартных сред, от CartPole до Atari и MuJoCo.
* [Hugging Face Deep RL Course](https://huggingface.co/learn/deep-rl-course/unit0/introduction): бесплатный курс с ноутбуками, можно обучить и выложить своего агента.
* [OpenAI Spinning Up](https://spinningup.openai.com/): короткие и чистые реализации основных алгоритмов + список ключевых статей.

## 2. Что такое обучение с подкреплением

**Reinforcement Learning (RL)** — раздел машинного обучения, в котором **агент** учится принимать решения, взаимодействуя со **средой**, чтобы максимизировать **суммарную награду**.

![paradigms](../../assets/ml_paradigms.png)

| | Supervised Learning | Unsupervised Learning | Reinforcement Learning |
|---|---|---|---|
| Данные | размеченные пары (x, y) | неразмеченные x | последовательность взаимодействий (s, a, r, s') |
| Обратная связь | правильный ответ сразу | нет обратной связи | награда, часто отложенная и зашумлённая |
| Данные i.i.d.? | да | да | нет: данные зависят от действий агента |
| Цель | минимизировать ошибку предсказания | найти структуру в данных | максимизировать суммарную награду |

Взаимодействие устроено как цикл:

![loop](../../assets/agent_env_loop.png)

На каждом шаге $t$:

1. агент наблюдает состояние $s_t$;
2. агент выбирает действие $a_t$ согласно своей **политике** $\pi(a \mid s)$;
3. среда переходит в новое состояние $s_{t+1}$ и выдаёт награду $r_{t+1}$;
4. повторить.

### Почему это сложнее, чем supervised learning

* **Нет правильного ответа.** Никто не говорит «здесь нужно было повернуть налево»; есть только число, и часто оно приходит с большой задержкой (выиграли партию через 200 ходов).
* **Credit assignment.** Какое из 200 действий было решающим? Награда одна на всех.
* **Данные порождает сам агент.** Плохая политика видит только плохие состояния. Распределение данных меняется по ходу обучения, привычные гарантии про i.i.d. не работают.
* **Exploration.** Чтобы найти лучшее поведение, нужно пробовать новое, а пробовать новое в среднем невыгодно. Про это раздел 6.

## 3. Живое демо: агент в среде Gymnasium

[Gymnasium](https://gymnasium.farama.org/) — стандартный интерфейс к средам: `env.reset()` возвращает первое наблюдение, `env.step(action)` — следующее наблюдение, награду и флаги окончания эпизода.

Возьмём классику: **CartPole**. Тележка едет по рельсу, на ней шест; нужно двигать тележку влево или вправо так, чтобы шест не упал. Наблюдение — 4 числа (положение и скорость тележки, угол и угловая скорость шеста), действий два. Награда +1 за каждый шаг, пока шест стоит; эпизод обрывается, когда шест отклонился больше 12° или тележка уехала за край.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym

env = gym.make("CartPole-v1")
obs, info = env.reset(seed=0)
print("observation_space:", env.observation_space)
print("action_space:     ", env.action_space)
print("первое наблюдение:", obs)

obs, reward, terminated, truncated, info = env.step(1)  # 1 = толкнуть вправо
print("после шага:        ", obs, "reward =", reward, "done =", terminated or truncated)

In [ ]:
# Посмотрим, как выглядит эпизод со случайными действиями: сохраним кадры.
env = gym.make("CartPole-v1", render_mode="rgb_array")
obs, _ = env.reset(seed=1)
frames = []
for t in range(60):
    frames.append(env.render())
    obs, r, terminated, truncated, _ = env.step(env.action_space.sample())
    if terminated or truncated:
        break
env.close()

idx = np.linspace(0, len(frames) - 1, 6).astype(int)
fig, axes = plt.subplots(1, 6, figsize=(16, 2.6))
for ax, i in zip(axes, idx):
    ax.imshow(frames[i])
    ax.set_title(f"t = {i}")
    ax.axis("off")
plt.suptitle(f"Случайная политика: шест падает за {len(frames)} шагов")
plt.show()

Случайная политика держит шест примерно 20 шагов. Придумаем **эвристику**: если шест падает вправо (угловая скорость положительная), толкаем тележку вправо, и наоборот. Это тоже политика, просто написанная руками.

In [ ]:
def random_policy(obs):
    return np.random.randint(2)

def heuristic_policy(obs):
    x, x_dot, theta, theta_dot = obs
    # толкаем тележку в ту сторону, куда сейчас падает шест
    return int(theta_dot > 0)

def run_episodes(policy, n_episodes=50, seed=0):
    env = gym.make("CartPole-v1")
    lengths = []
    for ep in range(n_episodes):
        obs, _ = env.reset(seed=seed + ep)
        total = 0
        while True:
            obs, r, terminated, truncated, _ = env.step(policy(obs))
            total += r
            if terminated or truncated:
                break
        lengths.append(total)
    env.close()
    return np.array(lengths)

np.random.seed(0)
for name, policy in [("случайная", random_policy), ("эвристика", heuristic_policy)]:
    L = run_episodes(policy)
    print(f"{name:10s}: средняя длина эпизода {L.mean():6.1f} ± {L.std():5.1f} (максимум 500)")

Эвристика примерно в 10 раз лучше случайной, но:

* её придумал человек, зная физику задачи;
* она не идеальна: до 500 шагов не дотягивает, тележка постепенно уезжает за край (попробуйте добавить в правило угол шеста и положение тележки);
* для шахмат, StarCraft или робота с 20 суставами такую эвристику руками не напишешь.

**Задача RL** — получить политику не хуже (а обычно лучше) эвристики, **не зная** устройства среды, только из опыта взаимодействия. На неделе 5 обучим на CartPole нейросеть (DQN), которая стабильно держит 500 шагов.

Если установлен `box2d`, можно посмотреть на среду посложнее: LunarLander, где нужно мягко посадить модуль на площадку.

In [ ]:
try:
    env = gym.make("LunarLander-v3", render_mode="rgb_array")
    obs, _ = env.reset(seed=0)
    frames = []
    for t in range(120):
        frames.append(env.render())
        obs, r, terminated, truncated, _ = env.step(env.action_space.sample())
        if terminated or truncated:
            break
    env.close()
    idx = np.linspace(0, len(frames) - 1, 4).astype(int)
    fig, axes = plt.subplots(1, 4, figsize=(16, 3.2))
    for ax, i in zip(axes, idx):
        ax.imshow(frames[i]); ax.set_title(f"t = {i}"); ax.axis("off")
    plt.suptitle("LunarLander, случайная политика: наблюдение из 8 чисел, 4 действия (двигатели)")
    plt.show()
except Exception as e:  # box2d не установлен или не собрался
    print("LunarLander недоступен:", type(e).__name__, "-", str(e)[:120])
    print("Гифку можно посмотреть в документации: https://gymnasium.farama.org/environments/box2d/lunar_lander/")

## 4. Из чего состоит задача RL

Зафиксируем словарь на примерах CartPole и маленького GridWorld.

![gridworld](../../assets/mdp_gridworld.png)

| Понятие | Обозначение | CartPole | GridWorld |
|---|---|---|---|
| **Состояние** (state) | $s \in \mathcal{S}$ | 4 числа: $x, \dot x, \theta, \dot\theta$ | номер клетки |
| **Действие** (action) | $a \in \mathcal{A}$ | {влево, вправо} | {↑, ↓, ←, →} |
| **Награда** (reward) | $r_{t+1} = R(s_t, a_t, s_{t+1})$ | +1 за каждый шаг | +1 выход, −1 яма, −0.04 за шаг |
| **Динамика** (transition) | $P(s' \mid s, a)$ | физика тележки | «скользкие» переходы |
| **Политика** (policy) | $\pi(a \mid s)$ | эвристика выше | стрелка в каждой клетке |
| **Эпизод** | $s_0, a_0, r_1, s_1, \ldots, s_T$ | пока шест не упал | пока не дошли до ямы/выхода |

Наблюдение и состояние — не всегда одно и то же: в покере вы видите свои карты, но не карты соперника. Такие задачи (POMDP) отложим до недели 13; пока считаем, что агент видит состояние целиком.

### Return и дисконтирование

Агент максимизирует не мгновенную награду, а **return** — суммарную награду с текущего шага:

$$
G_t = r_{t+1} + \gamma\, r_{t+2} + \gamma^2 r_{t+3} + \ldots = \sum_{k=0}^{\infty} \gamma^k r_{t+k+1}, \qquad \gamma \in [0, 1).
$$

Коэффициент дисконтирования $\gamma$:

* делает сумму конечной для бесконечных эпизодов;
* задаёт «горизонт планирования»: награда через $k$ шагов весит $\gamma^k$, эффективный горизонт $\approx 1/(1-\gamma)$;
* математически удобен (сжимающее отображение, пригодится на неделе 2).

In [ ]:
ks = np.arange(0, 200)
for gamma in [0.5, 0.9, 0.99]:
    plt.plot(ks, gamma ** ks, label=f"γ = {gamma}, горизонт ≈ {1/(1-gamma):.0f} шагов")
plt.xlabel("k (сколько шагов вперёд)")
plt.ylabel("вес награды γ^k")
plt.title("Дисконтирование: насколько агенту «важно» будущее")
plt.legend()
plt.show()

### Награда — это спецификация задачи, и её легко испортить

Агент оптимизирует ровно то, что написано в награде, а не то, что вы имели в виду. Классический пример — [CoastRunners](https://openai.com/index/faulty-reward-functions/): агент, обученный на очки, вместо прохождения трассы бесконечно крутится по кругу, собирая бонусы, и врезается в стены. Это называется **reward hacking**; обзор с десятками примеров есть у [Lilian Weng](https://lilianweng.github.io/posts/2024-11-28-reward-hacking/).

Практическое правило: награда должна описывать *что* нужно получить, а не *как* это делать. Если вы пишете в награду «держи угол шеста маленьким», агент найдёт способ держать угол маленьким, необязательно тот, который вы ожидали.

## 5. Exploration vs exploitation: многорукие бандиты

Прежде чем переходить к полной постановке, разберём задачу **без состояний** — **многорукий бандит** (multi-armed bandit). Она изолированно показывает главную дилемму RL.

Бытовые примеры той же задачи:

* выбираете ресторан на обед: пойти в проверенный или попробовать новый?
* A/B-тестирование и рекомендации: показывать баннер, который уже даёт клики, или тестировать новые?
* дозировки в клинических испытаниях, размещение рекламы, выбор маршрута.

### Постановка

* $K$ «рук» (действий), у руки $k$ своё неизвестное распределение награды с матожиданием $\mu_k$.
* На шаге $t$ агент выбирает руку $a_t$ и получает награду $r_t \sim P(r \mid a_t)$.
* Цель — максимизировать суммарную награду за $T$ шагов, то есть минимизировать **regret**:

$$
R_T = T \cdot \mu^* - \mathbb{E}\Big[\sum_{t=1}^{T} r_t\Big], \qquad \mu^* = \max_k \mu_k .
$$

Агент не знает $\mu_k$, их нужно оценивать по ходу дела. Отсюда дилемма:

* **Exploitation** — выбирать руку с лучшей текущей оценкой;
* **Exploration** — пробовать другие руки, чтобы уточнить оценки (вдруг они лучше).

### Стратегии

**ε-greedy.** С вероятностью $1-\varepsilon$ выбираем руку с максимальной оценкой $\hat Q(a)$, с вероятностью $\varepsilon$ — случайную. Просто и работает, но exploration «слепой»: агент одинаково часто трогает хорошо изученную плохую руку и плохо изученную.

**UCB1 (Upper Confidence Bound).** Выбираем руку по верхней доверительной границе:

$$
a_t = \arg\max_a \Big[ \hat{Q}(a) + c \sqrt{\frac{\ln t}{N(a)}} \Big],
$$

где $N(a)$ — сколько раз выбрали руку $a$. «Оптимизм перед лицом неопределённости»: чем меньше знаем про руку, тем больше бонус, и он убывает с накоплением статистики.

**Thompson Sampling.** Байесовский подход: храним апостериорное распределение над $\mu_k$ (например, Beta для Bernoulli-наград), на каждом шаге сэмплируем $\theta_k$ из него и выбираем руку с максимальным $\theta_k$. На практике часто не хуже UCB, а иногда лучше.

Все три стратегии при правильных гиперпараметрах дают regret $O(\log T)$, и это теоретический нижний предел (Lai & Robbins, 1985), но константы и поведение на малых $T$ различаются сильно. На семинаре реализуем все три и сравним.

In [ ]:
# Иллюстрация: чисто жадная стратегия (exploration = 0) может навсегда «залипнуть» на плохой руке.
rng = np.random.default_rng(0)
true_means = [0.2, 0.5, 0.55]  # руки: плохая, средняя, лучшая

def pure_greedy_regret(n_steps=500, n_init=1):
    Q = np.zeros(len(true_means))
    N = np.zeros(len(true_means))
    regret = np.zeros(n_steps)
    for a in range(len(true_means)):          # инициализация: n_init проб каждой руки
        for _ in range(n_init):
            r = rng.binomial(1, true_means[a])
            N[a] += 1
            Q[a] += (r - Q[a]) / N[a]
    for t in range(n_steps):
        a = np.argmax(Q)
        r = rng.binomial(1, true_means[a])
        N[a] += 1
        Q[a] += (r - Q[a]) / N[a]
        regret[t] = max(true_means) - true_means[a]
    return np.cumsum(regret)

for n_init in [1, 5, 20]:
    for run in range(5):
        plt.plot(pure_greedy_regret(n_init=n_init), color=f"C{[1,5,20].index(n_init)}",
                 alpha=0.7, label=f"n_init={n_init}" if run == 0 else None)
plt.xlabel("шаг t")
plt.ylabel("суммарный regret")
plt.title("Чисто жадная стратегия: 5 запусков на каждую инициализацию")
plt.legend()
plt.show()

При малом числе начальных проб жадная стратегия с заметной вероятностью «залипает» на неоптимальной руке: regret растёт **линейно**, а не логарифмически. Это простейшая иллюстрация того, почему чистый exploitation без exploration не гарантирует сходимости к оптимуму. В deep RL та же проблема выглядит как агент, который нашёл один способ получать маленькую награду и больше ничего не пробует.

## 6. Марковский процесс принятия решений (MDP)

Бандит — MDP с одним состоянием. Общая постановка задачи RL — **Markov Decision Process**, кортеж $(\mathcal{S}, \mathcal{A}, P, R, \gamma)$:

* $\mathcal{S}$ — множество состояний;
* $\mathcal{A}$ — множество действий;
* $P(s' \mid s, a)$ — вероятность перехода в $s'$ из $s$ при действии $a$;
* $R(s, a, s')$ — награда за переход (бывают варианты $R(s,a)$ и $R(s)$);
* $\gamma \in [0, 1)$ — коэффициент дисконтирования.

### Марковское свойство

$$
P(s_{t+1} \mid s_t, a_t, s_{t-1}, a_{t-1}, \ldots, s_0) = P(s_{t+1} \mid s_t, a_t).
$$

Будущее зависит от прошлого только через текущее состояние. Это не ограничение на реальность, а требование к тому, **как мы определяем состояние**: если нужно, в состояние включаем историю. В CartPole поэтому в наблюдении есть скорости, а в DQN на Atari в состояние складывают 4 последних кадра.

### Политика, ценность состояния и действия

* **Политика** $\pi(a \mid s)$ — распределение над действиями в состоянии $s$ (детерминированная политика — частный случай).
* **Value function** $V^\pi(s)$ — ожидаемый return при старте из $s$ и следовании $\pi$:

$$
V^\pi(s) = \mathbb{E}_\pi[G_t \mid S_t = s].
$$

* **Action-value function** $Q^\pi(s, a)$ — ожидаемый return при старте из $s$, выборе $a$ и следовании $\pi$ далее:

$$
Q^\pi(s, a) = \mathbb{E}_\pi[G_t \mid S_t = s, A_t = a].
$$

$V$ отвечает на вопрос «насколько хорошо здесь находиться», $Q$ — «насколько хорошо здесь сделать вот это». Почти все алгоритмы курса так или иначе оценивают одну из этих функций.

### Уравнения Беллмана

Ключевое наблюдение: return **рекурсивен**, $G_t = r_{t+1} + \gamma G_{t+1}$. Взяв матожидание, получаем уравнения Беллмана (ожидания):

$$
V^\pi(s) = \sum_a \pi(a \mid s) \sum_{s'} P(s' \mid s, a) \big[ R(s, a, s') + \gamma V^\pi(s') \big],
$$

$$
Q^\pi(s, a) = \sum_{s'} P(s' \mid s, a) \Big[ R(s, a, s') + \gamma \sum_{a'} \pi(a' \mid s')\, Q^\pi(s', a') \Big].
$$

Связь между ними:

$$
V^\pi(s) = \sum_a \pi(a \mid s)\, Q^\pi(s, a), \qquad
Q^\pi(s, a) = \sum_{s'} P(s' \mid s, a) \big[ R(s, a, s') + \gamma V^\pi(s') \big].
$$

Для **оптимальной** политики сумма по $a'$ превращается в $\max_{a'}$ — это уравнения оптимальности Беллмана, ими займёмся на неделе 2.

Эти уравнения — основа **всех** методов курса: динамическое программирование (неделя 2) решает их напрямую итерациями; TD-обучение (неделя 3) оценивает их по сэмплам; deep RL (недели 5+) аппроксимирует $V$ или $Q$ нейросетью и минимизирует невязку уравнения.

### Маленький пример: посчитаем $V^\pi$ руками и численно

Цепочка из трёх состояний. Из $s_0$ и $s_1$ политика всегда идёт «вправо», переход детерминированный, награда +1 за приход в $s_2$ (терминальное, $V(s_2) = 0$), остальные переходы дают 0. Тогда

$$
V^\pi(s_1) = 1 + \gamma \cdot 0 = 1, \qquad V^\pi(s_0) = 0 + \gamma V^\pi(s_1) = \gamma .
$$

Проверим, что уравнение Беллмана можно решать просто повторным применением правой части (это и есть iterative policy evaluation с недели 2).

In [ ]:
gamma = 0.9
# P[s, s'] при следовании политике «вправо»; s2 — терминальное (переход в себя с наградой 0)
P = np.array([[0, 1, 0],
              [0, 0, 1],
              [0, 0, 1]], dtype=float)
R = np.array([0.0, 1.0, 0.0])   # ожидаемая награда за шаг из s

V = np.zeros(3)
for it in range(30):
    V_new = R + gamma * P @ V
    V_new[2] = 0.0              # терминальное состояние
    if np.max(np.abs(V_new - V)) < 1e-10:
        break
    V = V_new
print(f"сошлось за {it} итераций: V = {V.round(4)}  (ожидаем [γ, 1, 0] = [{gamma}, 1, 0])")

## 7. Карта курса

![taxonomy](../../assets/rl_taxonomy.png)

Первые четыре недели — **табличные методы**: состояний мало, всё можно хранить в массиве, и на них видно устройство алгоритмов без шума нейросетей. Дальше те же идеи переносим на нейросети (**deep RL**), потом расширяем: обучение модели среды, exploration, обучение по логам без взаимодействия, несколько агентов, RL для языковых моделей.

![course](../../assets/course_map.png)

Всё, что в этой таблице, мы **реализуем сами** с нуля: от ε-greedy до PPO и SAC. Это принципиальная позиция курса: RL-алгоритмы печально известны тем, что «почти работающая» реализация не работает вовсе, и понять, где ошибка, можно только зная каждую строчку.

## 8. Литература и ссылки

**Базовые**

* R. Sutton, A. Barto. *Reinforcement Learning: An Introduction*, 2nd ed. — [бесплатный PDF](http://incompleteideas.net/book/the-book-2nd.html). Главы 1–3 покрывают сегодняшнюю лекцию.
* D. Silver. [UCL Course on RL](https://www.davidsilver.uk/teaching/) — 10 лекций с видео, классика.
* S. Levine. [CS285: Deep RL (Berkeley)](https://rail.eecs.berkeley.edu/deeprlcourse/) — про deep RL, пригодится с недели 5.
* [OpenAI Spinning Up](https://spinningup.openai.com/) — короткое введение + чистые реализации.

**Практика**

* [Gymnasium](https://gymnasium.farama.org/) — документация по средам и интерфейсу.
* [CleanRL](https://github.com/vwxyzjn/cleanrl) — каждый алгоритм в одном файле, удобно читать.
* [Stable-Baselines3](https://stable-baselines3.readthedocs.io/) — библиотека для быстрых экспериментов и сравнения со своей реализацией.
* [Hugging Face Deep RL Course](https://huggingface.co/learn/deep-rl-course/unit0/introduction).

**Статьи, упомянутые сегодня**

* Mnih et al. [Human-level control through deep RL](https://www.nature.com/articles/nature14236) (DQN), 2015.
* Silver et al. [Mastering the game of Go without human knowledge](https://www.nature.com/articles/nature24270) (AlphaGo Zero), 2017.
* Ouyang et al. [Training language models to follow instructions with human feedback](https://arxiv.org/abs/2203.02155) (InstructGPT), 2022.
* DeepSeek-AI. [DeepSeek-R1](https://arxiv.org/abs/2501.12948), 2025.

## На семинаре

* Интерфейс Gymnasium: `reset`, `step`, `action_space`, `observation_space` (см. `../seminar/seminar.ipynb`).
* Реализация среды бандита с нуля и агентов ε-greedy / UCB1 / Thompson Sampling, сравнение regret.
* Первое знакомство с табличной MDP-средой FrozenLake.
* **Мини-семинар по PyTorch** для тех, кто с ним не работал: `../seminar/pytorch_intro.ipynb`. Тензоры, autograd, `nn.Module`, цикл обучения и первый «агент на нейросети» для CartPole.

## Домашнее задание

См. `../homework/homework.ipynb`: бандиты (реализация и сравнение стратегий) и теория (return, вывод уравнения Беллмана, MDP на бумаге).